Environment Setup & Installations

In [1]:
# Cell 1: Modern Environment Setup
!pip install -q -U langchain-core langchain-huggingface gradio transformers accelerate

print("✅ Cell 1: Environment configured with the latest libraries.")

✅ Cell 1: Environment configured with the latest libraries.


AI Backend Initialization (Local LLM)

In [7]:
# Cell 2: Upgraded AI Backend Initialization (Using powerful Qwen 3B model)
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# Clean GPU memory
gc.collect()
torch.cuda.empty_cache()

# Upgraded to 3B Instruct model for superior intelligence and multilingual support
model_id = "Qwen/Qwen2.5-3B-Instruct"
print("⏳ Loading powerful Qwen 3B AI model...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024, # Larger tokens for complete roadmaps and explanations
    temperature=0.3,     # Lower temperature for accurate, factual, and direct answers
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_pipeline)
print("✅ Cell 2: Upgraded Qwen 3B Model loaded successfully!")

⏳ Loading powerful Qwen 3B AI model...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Cell 2: Upgraded Qwen 3B Model loaded successfully!


Advanced LCEL Agent, Memory & Web Interface

In [8]:
# Cell 3: Advanced LCEL Agent, Memory, and Strict System Instructions
import gradio as gr
from langchain_core.prompts import PromptTemplate

# Strict System Instructions & Formatting to prevent canned responses
template = """<|im_start|>system
You are an expert AI Engineer and Senior Technical Mentor.
Your goal is to provide comprehensive, accurate, professional, and direct answers to the user.
Never say "I cannot help you" if the user asks about technology, roadmaps, programming, or AI. Always provide practical and detailed guides.
<|im_end|>
{history}<|im_start|>user
{input}<|im_end|>
<|im_start|>assistant
"""

prompt = PromptTemplate.from_template(template)
ai_chain = prompt | llm

def process_chat_interaction(user_message, history):
    try:
        # Build clean conversation history for memory
        formatted_history = ""
        for item in history:
            if hasattr(item, 'role') and hasattr(item, 'content'):
                 if item.role == "user":
                     formatted_history += f"<|im_start|>user\n{item.content}<|im_end|>\n"
                 elif item.role == "assistant":
                     formatted_history += f"<|im_start|>assistant\n{item.content}<|im_end|>\n"
            elif isinstance(item, dict):
                if item.get("role") == "user":
                     formatted_history += f"<|im_start|>user\n{item.get('content')}<|im_end|>\n"
                elif item.get("role") == "assistant":
                     formatted_history += f"<|im_start|>assistant\n{item.get('content')}<|im_end|>\n"
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                formatted_history += f"<|im_start|>user\n{item[0]}<|im_end|>\n<|im_start|>assistant\n{item[1]}<|im_end|>\n"

        # Invoke the chain with memory and instructions
        ai_response = ai_chain.invoke({
            "history": formatted_history,
            "input": user_message
        })

        clean_response = ai_response.replace("<|im_end|>", "").strip()
        return clean_response
    except Exception as e:
        return f"⚠️ System Error: {str(e)}"

# Launch Web Interface
chatbot_app = gr.ChatInterface(
    fn=process_chat_interaction,
    title="🤖 Advanced AI Agent & Mentor (Qwen 3B)",
    description="Full-stack AI chatbot equipped with memory, strict system instructions, and advanced local intelligence.",
    examples=["Give me a roadmap to master AI Agents", "What is RAG architecture?", "Explain Python OOP in short"]
)

print("🚀 Starting Web Server with Upgraded Model...")
chatbot_app.launch(share=True, inline=True)

🚀 Starting Web Server with Upgraded Model...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4884f82e9538d9d915.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
